# Interaction Ratios vs Distance

Santi

09 April 2026

The goal of this notebook is to to plot Cs Rydberg state vs max interaction at fixed distances (for both NaCs and RbCs). Additionally, we want to plot the optimal ratio between inter- and intra-species interactions possible by changing the atom spacings.

In [ ]:
# import two_atom_interaction.py
from two_atom_interaction import *
# keep track of time
import time

## Quick test.

This is how we will calculate the interaction energy for the various distances

In [ ]:
# check the C6 coefficient value with exact diagonalization using ARC
atom1 = Sodium()
atom2 = Cesium()
n_atom1 = 51
l_atom1 = 0
j_atom1 = 0.5
n_atom2 = 54
l_atom2 = 0
j_atom2 = 0.5

# Build inter-species pair-state calculation
calc = PairStateInteractions(
    atom1, n_atom1, l_atom1, j_atom1,
    n_atom2, l_atom2, j_atom2,
    m1=0.5, m2=0.5, atom2=atom2
)
calc.defineBasis(0, 0, 5, 3, 25e9, progressOutput=True)

R_um = 5
calc.diagonalise([R_um], 200, progressOutput=True)

# find state with highest overlap with target state
j = np.argmax(np.abs(calc.highlight[0]))
print("shift at", R_um, "um =", calc.y[0][j], "GHz")

In [ ]:
# define a function to calculate the energy shift for a given R
def calc_energy_shift(cfg: ExperimentConfig, R_um: float) -> float:
    atom1 = cfg.atom1
    n_atom1 = cfg.n_atom1
    l_atom1 = cfg.l_atom1
    j_atom1 = cfg.j_atom1
    atom2 = cfg.atom2
    n_atom2 = cfg.n_atom2
    l_atom2 = cfg.l_atom2
    j_atom2 = cfg.j_atom2

    # Build inter-species pair-state calculation
    calc = PairStateInteractions(
        atom1, n_atom1, l_atom1, j_atom1,
        n_atom2, l_atom2, j_atom2,
        m1=0.5, m2=0.5, atom2=atom2
    )
    calc.defineBasis(0, 0, 5, 3, 25e9, progressOutput=True)

    calc.diagonalise([R_um], 100, progressOutput=True)

    # find state with highest overlap with target state
    j = np.argmax(np.abs(calc.highlight[0]))
    energy_shift_GHz = calc.y[0][j]
    return energy_shift_GHz

## Sweeping over Rb-Cs and Na-Cs states at fixed R = 5 um

In [ ]:
# perform sweep over states for Rb-Cs and Na-Cs at R = 5 um
Cs_states = np.linspace(50, 90, 41, dtype=int)  # n=50 to 90
energy_shifts_RbCs = []
energy_shifts_NaCs = []
for i in range(len(Cs_states)):
    Rb_states = np.linspace(Cs_states[i]-5, Cs_states[i]+5, 11, dtype=int)
    Na_states = np.linspace(Cs_states[i]-5, Cs_states[i]+5, 11, dtype=int)
    for Rb_state in Rb_states:
        cfg_temp = ExperimentConfig(
            atom1 = Rubidium(),
            mass_atom1 = Rubidium().mass,
            n_atom1 = Rb_state,  # Rb Rydberg n

            atom2 = Cesium(),
            mass_atom2 = Cesium().mass,
            n_atom2 = Cs_states[i],  # Cs Rydberg n
        )
        energy_shift = calc_energy_shift(cfg_temp, R_um=5)
        energy_shifts_RbCs.append(energy_shift)
    for Na_state in Na_states:
        cfg_temp = ExperimentConfig(
            atom1 = Sodium(),
            mass_atom1 = Sodium().mass,
            n_atom1 = Na_state,  # Na Rydberg n

            atom2 = Cesium(),
            mass_atom2 = Cesium().mass,
            n_atom2 = Cs_states[i],  # Cs Rydberg n
        )
        energy_shift = calc_energy_shift(cfg_temp, R_um=5)
        energy_shifts_NaCs.append(energy_shift)

# plot energy shifts for Rb-Cs and Na-Cs
plt.figure(figsize=(10,6))
plt.scatter(np.repeat(Cs_states, 11), energy_shifts_RbCs, color='red', label='Rb-Cs')
plt.scatter(np.repeat(Cs_states, 11), energy_shifts_NaCs, color='blue', label='Na-Cs')
plt.xlabel('Cs Rydberg State (n)')
plt.ylabel('Energy Shift (GHz)')
plt.title('Energy Shifts for Rb-Cs and Na-Cs')
plt.legend()
plt.show()

In [ ]:
# create a new txt file and save the arrays of data to it (add data and time to each file)
import datetime

# Get the current date and time
now = datetime.datetime.now()

# Create a new text file with a unique name
file_name = f"energy_shifts_{now.strftime('%Y%m%d_%H%M%S')}.txt"
with open(file_name, 'w') as f:
    f.write("Cs Rydberg State (n), Rb-Cs Energy Shift (GHz), Na-Cs Energy Shift (GHz)\n")
    for i in range(len(Cs_states)):
        for j in range(11):
            f.write(f"{Cs_states[i]}, {energy_shifts_RbCs[i*11+j]}, {energy_shifts_NaCs[i*11+j]}\n")

In [ ]:
# plot the max energy shifts for each Cs state for Na and Rb
# find the max energy shifts for each Cs state
max_energy_shifts_RbCs = []
max_energy_shifts_NaCs = []
for i in range(len(Cs_states)):
    Rb_shifts = energy_shifts_RbCs[i*11:(i+1)*11]
    Na_shifts = energy_shifts_NaCs[i*11:(i+1)*11]
    max_energy_shifts_RbCs.append(np.max(np.abs(Rb_shifts)))
    max_energy_shifts_NaCs.append(np.max(np.abs(Na_shifts)))

# plot the max energy shifts and the corresponding Na and Rb states on a different plot side by side
plt.figure(figsize=(12,6))
plt.subplot(1, 2, 1)
plt.plot(Cs_states, max_energy_shifts_NaCs, marker='o', label='Na-Cs')
plt.plot(Cs_states, max_energy_shifts_RbCs, marker='o', label='Rb-Cs')
plt.xlabel("Cs Rydberg state n", fontsize=14)
plt.ylabel("Max absolute Energy Shift (GHz)", fontsize=14)
plt.title("Max Energy Shifts for Na/Rb - Cs Interaction")
plt.yscale('log')
plt.legend()
plt.grid()

plt.subplot(1, 2, 2)
# find the n values that give the max energy shifts
max_n_Na_energy = []
max_n_Rb_energy = []
for i in range(len(Cs_states)):
    Rb_shifts = energy_shifts_RbCs[i*11:(i+1)*11]
    Na_shifts = energy_shifts_NaCs[i*11:(i+1)*11]
    Rb_states = np.linspace(Cs_states[i]-5, Cs_states[i]+5, 11, dtype=int)
    Na_states = np.linspace(Cs_states[i]-5, Cs_states[i]+5, 11, dtype=int)
    max_n_Rb_energy.append(Rb_states[np.abs(Rb_shifts).argmax()])
    max_n_Na_energy.append(Na_states[np.abs(Na_shifts).argmax()])
plt.plot(Cs_states, max_n_Na_energy, marker='o', label='Na')
plt.plot(Cs_states, max_n_Rb_energy, marker='o', label='Rb')
plt.xlabel("Cs Rydberg state n", fontsize=14)
plt.ylabel("Rydberg state n giving max Energy Shift", fontsize=14)
plt.title("Rydberg state n giving Max Energy Shift for Na/Rb - Cs Interaction")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

## Now sweeping R and finding interaction energy for Rb-Cs and Na-Cs

Here we will fix the state to be the one for the maximum interaction energy from above. 